# Traceability: ledger, artifacts, manifest, run log

Every number below is read from an aggregate result artifact in `../artifacts/` — the same files the paper's tables and figures were generated from.
This notebook reads only `../artifacts/` and `../ARTIFACT_MANIFEST.csv`.


## Claims ledger → artifacts
Every row of `CLAIMS_LEDGER.csv` names an artifact file and a JSON path; this cell resolves each and compares the stored value (tolerance 5e-5 or 1e-6 relative). Object-valued rows point at whole result blocks (a table row, a CI pair) and are reported as derived.


In [ ]:
import json, os, csv
ART = "../artifacts"
rows = list(csv.DictReader(open(os.path.join(ART, "CLAIMS_LEDGER.csv"), encoding="utf-8-sig")))
def resolve(o, path):
    for k in [k for k in path.split(".") if k]:
        o = o[int(k)] if isinstance(o, list) else o[k]
    return o
exact = derived = mismatch = missing = shawarn = 0; bad = []
for r in rows:
    f = os.path.join(ART, os.path.basename(r["source_json"]))
    if not os.path.exists(f):
        missing += 1; bad.append((r["claim_id"], "artifact not in repository", os.path.basename(r["source_json"]))); continue
    d = json.load(open(f, encoding="utf-8"))
    if r["sha256_16"] and d.get("sha256_16") and r["sha256_16"] != d["sha256_16"]: shawarn += 1
    try: o = resolve(d, r["json_path"])
    except Exception: mismatch += 1; bad.append((r["claim_id"], "path does not resolve", r["json_path"][:70])); continue
    if isinstance(o, (dict, list)): derived += 1; continue
    try: ok = abs(float(r["value"]) - float(o)) <= max(5e-5, abs(float(o)) * 1e-6)
    except Exception: ok = str(o) == r["value"]
    if ok: exact += 1
    else: mismatch += 1; bad.append((r["claim_id"], "value", f"{r['value']} vs {o}"))
print(f"ledger rows {len(rows)} · exact value match {exact} · object-valued (derived) rows {derived} · mismatches {mismatch} · artifact missing {missing} · script-hash warnings {shawarn}")
for b in bad[:20]: print("  ", b)
from collections import Counter
print("status:", dict(Counter(r["path_status"] for r in rows)))
assert mismatch == 0 and missing == 0, "ledger does not resolve against the artifacts"


ledger rows 726 · exact value match 629 · object-valued (derived) rows 97 · mismatches 0 · artifact missing 0 · script-hash warnings 0
status: {'valid': 711, 'valid (v1 전시물 — v2 robustness 절 참조)': 3, 'valid (v2 정본 재지정)': 4, 'valid (v2: robustness 행)': 2, 'valid (단위 정정)': 1, 'valid (canonical: P00106)': 1, 'SUPERSEDED': 4}


## Artifact hashes → manifest
`ARTIFACT_MANIFEST.csv` records the SHA-256 prefix of every artifact at assembly time.


In [ ]:
import hashlib, csv
man = list(csv.DictReader(open("../ARTIFACT_MANIFEST.csv", encoding="utf-8")))
bad = [m["file"] for m in man if hashlib.sha256(open(os.path.join(ART, m["file"]), "rb").read()).hexdigest()[:16] != m["sha256_16"]]
print(f"artifacts in manifest {len(man)} · hash mismatches {len(bad)}", bad[:5])
assert not bad
cited = sum(int(m["ledger_rows"]) > 0 for m in man); gen = sum(m["read_by_exhibit_generator"] == "yes" for m in man)
print(f"artifacts cited by the ledger {cited} · read by the exhibit generator {gen}")


artifacts in manifest 82 · hash mismatches 0 []
artifacts cited by the ledger 71 · read by the exhibit generator 59


## Run log
One row per harness run, with the SHA-256 prefix of the script as run.


In [ ]:
runs = list(csv.DictReader(open(os.path.join(ART, "run_log.csv"), encoding="utf-8-sig")))
print(f"harness runs logged: {len(runs)}")
for r in runs[-8:]: print(f"  {r['run_id']:<9} {r['date']}  {r['script']:<44} {r['verdict']}")


harness runs logged: 62
  P001-56   2026-09-09  p001_56_within_round_v2.py                   OK
  P001-52   2026-09-09  p001_52_balance_maxt.py                      OK
  P001-54   2026-09-09  p001_54_within_partner_outcome.py            GO
  P001-57   2026-09-09  p001_57_within_partner_horizon.py            OK
  P001-55   2026-09-09  p001_55_table3_fixed_horizon_ladder.py       OK
  P001-58   2026-09-09  p001_58_public_sources_controls.py           OK
  P001-59   2026-09-09  p001_59_within_partner_extensions.py         OK
  P001-60   2026-09-10  p001_60_review_reanalyses.py                 OK
